In [1]:
import json

base_path = "../results"

models = [
    "gemini-2.0-flash",
    "microsoft/phi-4",
    "openai/gpt-4.1-mini",
    "anthropic/claude-3-haiku",
    "deepseek/deepseek-chat-v3-0324:free",
    "mistralai/mistral-small-3.1-24b-instruct",
    "meta-llama/llama-3.3-70b-instruct",
    "qwen/qwen-2.5-72b-instruct"
]

In [7]:
def get_success_rate(samples):
    success = 0
    for sample in samples:
        if "new cost" in sample:
            success += 1
    return success / len(samples)

In [12]:
import numpy as np 

def get_avg_cost_reduction(samples):
    reductions = []
    for sample in samples:
        if "new cost" in sample:
            reductions.append(sample["current cost"] - sample["new cost"])
        else:
            reductions.append(0)
    return np.mean(reductions)

In [5]:
def get_avg_resp_time(samples):
    resp_times = []
    for sample in samples:
        resp_times.append(sample["response time"])
    return np.mean(resp_times)

In [8]:
def get_avg_num_queries(samples):
    num_queries = []
    for sample in samples:
        if sample["attempt"] != -1:
            num_queries.append(sample["attempt"])
    
    return np.mean(num_queries)

In [15]:
for model in models:
    print(model)
    all_samples = []
    for sample_idx in range(10):
        sample_path = f"{base_path}/{model}/{sample_idx}.json"
        with open(sample_path, "r") as f:
            sample_result = json.load(f)
        all_samples.append(sample_result)
    print(len(all_samples))
    for sample in all_samples:
        if "target feature" in sample:
            print(sample["target feature"], end=", ")
    print()
    success_rate = get_success_rate(all_samples)
    avg_cost_reduction = get_avg_cost_reduction(all_samples)
    avg_resp_time = get_avg_resp_time(all_samples)
    avg_num_queries = get_avg_num_queries(all_samples)  

    print(f"success_rate: {success_rate}")  
    print(f"avg_cost_reduction: {avg_cost_reduction}")  
    print(f"avg_resp_time: {avg_resp_time}")  
    print(f"avg_num_queries: {avg_num_queries}")  


gemini-2.0-flash
10
Facility Name, Facility Name, Facility Name, Facility Name, Facility Name, Facility Name, Facility Name, Facility Name, Facility Name, Facility Name, 
success_rate: 1.0
avg_cost_reduction: 10291.171110534668
avg_resp_time: 46.94604642391205
avg_num_queries: 1.3
microsoft/phi-4
10
Facility Name, Type of Admission, Facility Name, Age Group, Facility Name, Facility Name, Facility Name, Facility Name, Payment Typology 1, Facility Name, 
success_rate: 1.0
avg_cost_reduction: 8890.757225036621
avg_resp_time: 251.94494774341584
avg_num_queries: 1.8888888888888888
openai/gpt-4.1-mini
10
Facility Name, Facility Name, 
success_rate: 0.2
avg_cost_reduction: 6004.980546569825
avg_resp_time: 142.9494750738144
avg_num_queries: 3.0
anthropic/claude-3-haiku
10

success_rate: 0.0
avg_cost_reduction: 0.0
avg_resp_time: 192.4156820297241
avg_num_queries: 2.25
deepseek/deepseek-chat-v3-0324:free
10
Facility Name, Facility Name, Facility Name, Facility Name, Patient Disposition, 
succes

In [19]:
EVALUATOR_SYSTEM_PROMPT = '''
You are a medical expert and health policy analyst.

Your task is to validate cost-reduction strategies proposed for a specific inpatient case. You will receive:
- The structured clinical and administrative data for the patient.
- SHAP values quantifying how much each feature contributes to the total cost.
- A list of strategies proposed by another model to reduce cost.

You must evaluate whether each strategy is:
- **Medically feasible** for the given condition.
- **Administratively actionable** within standard hospital operations.
- **Non-hallucinated** (i.e., not fabricated or clinically unsafe).
- **Consistent with the data provided**, especially with feature values and SHAP importance.

Only use the information provided. Do not assume facts not in evidence.
Return your answer in structured JSON format.
'''

In [ ]:
EVALUATOR_USER_PROMPT = f'''
You are given:
1. The structured data for a patient.
2. SHAP values for each feature.
3. Proposed strategies to reduce cost.

Your tasks:
- For each strategy, assess:
  - Is it medically valid? (Yes/No)
  - Is it administratively feasible? (Yes/No)
  - Does it seem hallucinated or unsafe? (Yes/No)
  - Optional: a short justification (max 2 sentences)

Respond in the following format only (no extra commentary):

{{
  "Feature Name 1": {{
    "Medically Valid": "Yes/No",
    "Administratively Feasible": "Yes/No",
    "Hallucinated": "Yes/No",
    "Justification": "..."
  }},
  ...
}}
  
Patient Data:
###patient_data###

SHAP Values:
###shap_values###

Proposed Strategies:
###llm_strategies###
'''

In [18]:
from openai import OpenAI
import os

or_api_key = os.getenv("OR_API_KEY")
client = OpenAI(
  base_url="https://openrouter.ai/api/v1",
  api_key=or_api_key,
)

In [ ]:
g_eval = {}
for model in models[6:]:
    print(model)
    g_eval[model] = {}
    all_samples = []
    for sample_idx in range(10):
        print(sample_idx)
        sample_path = f"{base_path}/{model}/{sample_idx}.json"
        with open(sample_path, "r") as f:
            sample_result = json.load(f)
        patient_data = sample_result.get("patient_dict", {})
        shap_values = sample_result.get("shap_info", {})
        llm_strategies = sample_result.get("suggested strategies", {})
        EVALUATOR_USER_PROMPT_updated = EVALUATOR_USER_PROMPT.replace("###patient_data###", json.dumps(patient_data, indent=2))
        EVALUATOR_USER_PROMPT_updated = EVALUATOR_USER_PROMPT_updated.replace("###shap_values###", json.dumps(shap_values, indent=2))
        EVALUATOR_USER_PROMPT_updated = EVALUATOR_USER_PROMPT_updated.replace("###llm_strategies###", json.dumps(llm_strategies, indent=2))
        completion = client.chat.completions.create(
            model="openai/gpt-4o-mini",
            messages=[
                {
                    "role": "system",
                    "content": EVALUATOR_SYSTEM_PROMPT
                },
                {
                    "role": "user",
                    "content": EVALUATOR_USER_PROMPT_updated
                },
            ]
        )
        g_eval[model][sample_idx] = json.loads(completion.choices[0].message.content)

In [30]:
with open(f"{base_path}/G_Eval.json", "w", encoding='utf8') as f:
    json.dump(g_eval, f, ensure_ascii=False, indent=4)

In [38]:
with open(f"{base_path}/G_Eval.json", "r") as f:
    g_eval = json.load(f)

for model, samples in g_eval.items():
    print(model)
    total = 0
    hallucinated = 0
    medically_valid = 0
    admin_feasibility = 0
    for sample_id, res in samples.items():
        if res:
            for target, info in res.items():
                total += 1
                if info["Hallucinated"].lower() == "yes":
                    hallucinated += 1
                if info["Medically Valid"].lower() == "yes":
                    medically_valid += 1
                if info["Administratively Feasible"].lower() == "yes":
                    admin_feasibility += 1
    print(hallucinated*100 / total if total > 0 else 0.0)
    print(medically_valid*100 / total if total > 0 else 0.0)
    print(admin_feasibility*100 / total if total > 0 else 0.0)

gemini-2.0-flash
23.008849557522122
42.47787610619469
51.32743362831859
microsoft/phi-4
0.0
78.35051546391753
96.90721649484536
openai/gpt-4.1-mini
0.0
87.09677419354838
96.7741935483871
anthropic/claude-3-haiku
0.0
86.48648648648648
97.29729729729729
deepseek/deepseek-chat-v3-0324:free
0.0
92.10526315789474
94.73684210526316
mistralai/mistral-small-3.1-24b-instruct
11.428571428571429
35.23809523809524
52.38095238095238
meta-llama/llama-3.3-70b-instruct
3.7037037037037037
54.93827160493827
61.111111111111114
qwen/qwen-2.5-72b-instruct
1.2658227848101267
77.21518987341773
89.87341772151899
